## PyCrown

In [ ]:
import sys
from datetime import datetime
from pycrown import PyCrown

In [ ]:
F_CHM = 'data/CHM.tif'
F_DTM = 'data/DTM.tif'
F_DSM = 'data/DSM.tif'
F_LAS = 'data/POINTS.las'

In [ ]:
PC = PyCrown(F_CHM, F_DTM, F_DSM, F_LAS, outpath='result')

In [ ]:
PC.clip_data_to_bbox((1802150, 1802408, 5467305, 5467480))

In [ ]:
PC.filter_chm(5, ws_in_pixels=True, circular=False)

In [ ]:
PC.tree_detection(PC.chm, ws=5, hmin=16.)

In [ ]:
PC.clip_trees_to_bbox(bbox=(1802160, 1802400, 5467315, 5467470))

In [ ]:
PC.crown_delineation(algorithm='dalponteCIRC_numba', th_tree=15.,
                     th_seed=0.7, th_crown=0.55, max_crown=10.)

In [ ]:
PC.correct_tree_tops()

In [ ]:
PC.get_tree_height_elevation(loc='top')
PC.get_tree_height_elevation(loc='top_cor')

In [ ]:
PC.screen_small_trees(hmin=20., loc='top')

In [ ]:
PC.crowns_to_polys_raster()
PC.crowns_to_polys_smooth(store_las=True)

In [ ]:
PC.quality_control()

In [ ]:
print(f"Number of trees detected: {len(PC.trees)}")

In [ ]:
PC.export_raster(PC.chm, PC.outpath / 'chm.tif', 'CHM')
PC.export_tree_locations(loc='top')
PC.export_tree_locations(loc='top_cor')
PC.export_tree_crowns(crowntype='crown_poly_raster')
PC.export_tree_crowns(crowntype='crown_poly_smooth')

## Alternative

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Function to generate sample data (replace this with your actual data)
def generate_sample_data(n_samples=1000):
    np.random.seed(42)
    x = np.random.uniform(0, 100, n_samples)
    y = np.random.uniform(0, 100, n_samples)
    z = np.random.uniform(0, 30, n_samples)
    
    # Simulating different tree species based on height (z)
    species = np.where(z < 10, 'Oak', np.where(z < 20, 'Pine', 'Maple'))
    
    return np.column_stack((x, y, z)), species

# Generate sample data
X, y = generate_sample_data()

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the k-NN classifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Make predictions on the test set
y_pred = knn.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Function to identify a tree species given its XYZ coordinates
def identify_tree_species(x, y, z):
    coordinates = np.array([[x, y, z]])
    species = knn.predict(coordinates)[0]
    return species

# Example usage
sample_tree = (50, 50, 15)
predicted_species = identify_tree_species(*sample_tree)
print(f"\nA tree at coordinates {sample_tree} is predicted to be a {predicted_species}.")
